In [0]:
with
-- parse workspaces json
workspace as (
  select explode(
    map_entries(from_json('$$$__WORKSPACES_JSON__$$$', 'map<string,string>'))
  ) as kvp,
  kvp['key'] as workspace_id,
  kvp['value'] as workspace_name
),
-- apply date filter
usage_with_ws_filtered_by_date as (
  select
    case
      when workspace_name is null then concat('id: ', u.workspace_id)
      else concat(workspace_name, ' (id: ', u.workspace_id, ')')
    end as workspace,
    u.*
  from system.billing.usage as u
  left join workspace
    on u.workspace_id = workspace.workspace_id
  where u.usage_date between :time_range.min and :time_range.max
),

-- apply workspace filter
usage_filtered as (
  select
    *
  from usage_with_ws_filtered_by_date
  where if(:param_workspace='<ALL WORKSPACES>', true, workspace = :param_workspace) -- all workspaces under account, or single workspace
    AND 
    -- Product SKU filter
  (array_contains(:product_category, billing_origin_product) OR array_contains(:product_category,'all'))
),

-- Add discounts dynamically
parsed_discounts_table AS (
    WITH split_data AS (
      SELECT 
        split(regexp_replace(upper(:discounts_by_product), '\\s+', ''), ';') AS kv_pairs
    ),
    exploded_data AS (
      SELECT 
        monotonically_increasing_id() AS order_id,
        explode(kv_pairs) AS kv_pair
      FROM 
        split_data
    ),

clean_keys AS (

      SELECT 
        split(kv_pair, '=')[0] AS product,
        try_cast(COALESCE(get(split(kv_pair, '='),1)::decimal(10,3), 0) AS decimal(10,3)) as discount,
        kv_pair AS combination,
        CASE WHEN contains(kv_pair, '=') THEN 1 ELSE 0 END AS ContainsValuePair
      FROM 
        exploded_data
    )
    SELECT * FROM clean_keys WHERE ContainsValuePair = 1
),

-- calc list priced usage in USD
prices as (
  select coalesce(price_end_time, date_add(current_date, 1)) as coalesced_price_end_time, *
  from system.billing.list_prices
  where currency_code = 'USD'
),

list_priced_usd as (
  select
    coalesce(((1-COALESCE(discounts.discount, 0))* p.pricing.default)*usage_quantity, u.usage_quantity * p.pricing.default, 0) as usage_usd,
    date_trunc('YEAR', usage_date) as usage_year,
    date_trunc('QUARTER', usage_date) as usage_quarter,
    date_trunc('MONTH', usage_date) as usage_month,
    date_trunc('WEEK', usage_date) as usage_week,
    u.*
  from usage_filtered as u
  LEFT JOIN parsed_discounts_table AS discounts ON discounts.product = u.billing_origin_product
  left join prices as p
    on u.sku_name=p.sku_name
    and u.usage_unit=p.usage_unit
    and (u.usage_end_time between p.price_start_time and p.coalesced_price_end_time)
),
-- eval time_key param
list_priced_usd_with_time_key as (
  select
    identifier
    (
      case
        when :param_time_key = 'Year' then 'usage_year'
        when :param_time_key = 'Quarter' then 'usage_quarter'
        when :param_time_key = 'Month' then 'usage_month'
        when :param_time_key = 'Week' then 'usage_week'
        when :param_time_key = 'Day' then 'usage_date'
        else 'usage_date'
      end
    )::date as time_key,
    *
  from list_priced_usd
),
-- eval group_key param
ws_count as (
  select
    count(distinct(workspace)) as workspace_count
  from list_priced_usd_with_time_key
  where workspace is not null
),
top_workspace_usage as (
  select
    workspace as top_workspace,
    sum(usage_usd) as _top_ws_usage_usd
  from list_priced_usd_with_time_key
  where workspace is not null
  group by top_workspace
  order by _top_ws_usage_usd desc
  limit 10
),
list_priced_usd_with_time_and_group_keys as (
  select
    if(workspace_count <= 50 or workspace is null or top_workspace is not null, workspace, '<OTHERS>') as workspace_norm,
    identifier
    (
      case
        when :param_group_key = 'Workspace' then 'workspace_norm'
        when :param_group_key = 'SKU' then 'sku_name'
        else 'billing_origin_product'
      end
    ) as group_key,
    *,
    CASE WHEN product_features.is_serverless = True THEN 'Serverless' ELSE 'Classic' END AS IsServerless
  from ws_count, list_priced_usd_with_time_key u
  left join top_workspace_usage
    on u.workspace = top_workspace_usage.top_workspace
)
-- query
select
  time_key, group_key, usage_usd, IsServerless
from list_priced_usd_with_time_and_group_keys
  WHERE (:is_serverless = 'All' 
    OR IsServerless = :is_serverless
    )
